# CVID spotting comparison v2 — runner template, R5 results audited

Kết quả R5 đã hoàn tất và audit16/10/2026: PGNet E2E45,60%; docTR38,13%; no_selection. Đây là template không outputs cho reproducibility/resume; notebook đã chạy và reviewed ở experiments/outputs/cvid-r5-final-audit-261016/notebook/. Giữ bundle40291c... và RUN_ID r5 chỉ khi resume đúng runtime identity. Đừng Run all chỉ để xem kết quả.


R5: preflight và warm-up trước test dùng chung inference + mở ảnh + geometry checks, rồi mới so toàn bộ runtime/native hashes. Sau ALL6 preflight có ALL6 train replay trong process mới; chưa đọc ảnh test. Mismatch lưu binding diff và cả expected/actual runtime dưới diagnostics và hiển thị trong notebook. Giữ r4 evidence/EasyOCR riêng; source/bundle identity đã đổi nên không chép records r4 sang r5. R5 cần preflight/replay mới và test mới. R5 T4 gates/inference/scoring/plots đã được audit16/10; giữ no_selection.


In [ ]:
from pathlib import Path
import sys, os, json, subprocess, hashlib, zipfile, shutil, time, uuid
from IPython.display import display, HTML
from html import escape
from google.colab import drive, files
drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/OCR')  # explicit persistent destination
RUN_ID = 'cvid-spotting-t4-261015-v2-r5'
CACHE_ID = 'cvid-spotting-t4-261015-v2-r2'  # reuse exact wheels/seals, never old run receipts
BUNDLE_ZIP = DRIVE_ROOT / 'iiit-cvid-spotting-colab-t4-inputs-v2.zip'
SOURCE_ZIP = None  # optional existing source ZIP path on Drive; otherwise pinned HF download
WORK = Path('/content/cvid-spotting-t4')
ROOT = WORK / 'runner'
OUT = DRIVE_ROOT / RUN_ID
# Keep exact wheels and package seals across Colab VM loss, outside evidence ZIP.
os.environ['SPOTTING_CACHE_ROOT'] = str(DRIVE_ROOT / (CACHE_ID + '-runtime-cache'))
WORK.mkdir(parents=True, exist_ok=True)
OUT.mkdir(parents=True, exist_ok=True)
gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,uuid,memory.total,driver_version',
                      '--format=csv,noheader'], check=True, text=True, capture_output=True)
print(gpu.stdout)
if 'T4' not in gpu.stdout:
    raise RuntimeError('This profile requires a T4. Select T4 and restart; do not reuse another GPU profile.')

print('OUT:',OUT,'Disk free GB:',round(shutil.disk_usage('/content').free/1e9,2))


## 1. Upload / verify comparison bundle hiện hành
Nếu ZIP chưa có trên Drive, cell hỏi upload từ máy. SHA và mọi member được kiểm trước khi gọi runner. Không upload qualification ZIP vào bước này. Tên ZIP giữ cố định; tải lại bản khớp notebook hiện hành.


In [ ]:
EXPECTED_BUNDLE_SHA256 = '40291c603b60eb564a66a182d12ae50cf4b19579954fac5c69c22d87c8ae57df'
def file_sha(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()
if not BUNDLE_ZIP.is_file():
    uploaded = files.upload()
    assert len(uploaded)==1, 'Upload đúng một comparison bundle.'
    upload_name, upload_bytes = next(iter(uploaded.items()))
    assert hashlib.sha256(upload_bytes).hexdigest()==EXPECTED_BUNDLE_SHA256, 'ZIP không khớp notebook.'
    BUNDLE_ZIP.parent.mkdir(parents=True,exist_ok=True)
    BUNDLE_ZIP.write_bytes(upload_bytes)
if file_sha(BUNDLE_ZIP) != EXPECTED_BUNDLE_SHA256:
    raise ValueError('Bundle hash differs from this notebook. Use the matching notebook/ZIP pair.')
from pathlib import PurePosixPath
with zipfile.ZipFile(BUNDLE_ZIP) as archive:
    names = archive.namelist()
    if len(names) != len(set(names)):
        raise ValueError('Duplicate ZIP members')
    for name in names:
        if PurePosixPath(name).is_absolute() or '..' in PurePosixPath(name).parts or '\\' in name or ':' in name:
            raise ValueError('Unsafe ZIP member')
    manifest = json.loads(archive.read('bundle-manifest.json'))
    entries = manifest['files']
    if len(entries) != len({e['path'] for e in entries}) or set(names) != {e['path'] for e in entries} | {'bundle-manifest.json'}:
        raise ValueError('Bundle membership changed')
    for entry in entries:
        value = archive.read(entry['path'])
        if len(value) != entry['bytes'] or hashlib.sha256(value).hexdigest() != entry['sha256']:
            raise ValueError('Bundle file changed: ' + entry['path'])
    # Use a fresh source directory to avoid stale modules outside the manifest.
    if ROOT.exists():
        shutil.rmtree(ROOT)
    archive.extractall(ROOT)
sys.path.insert(0, str(ROOT))
print('Verified bundle files:', len(entries))

from experiments.scripts.spotting_colab_candidates import load_registry
from experiments.scripts.spotting_colab_review import table, status_rows, require_preflights, require_train_replays
from experiments.scripts.spotting_colab_state import read
REGISTRY_PATH = ROOT/'experiments/registry/datasets/iiit-cvid-spotting-colab-t4-candidates-261015-v2.json'
registry = load_registry(REGISTRY_PATH)
assert len(registry['candidates'])==6 and all(not c.get('conditional',False) for c in registry['candidates'])
table([dict(candidate_id=c['candidate_id'],environment=c['environment'],locked=c['locked'],configuration_sha256=c['configuration_sha256']) for c in registry['candidates']], 'Six locked candidates')
table([read(ROOT/'experiments/registry/datasets/iiit-cvid-pgnet-t4-qualification-audit-261015.json')], 'PGNet qualification audited — not comparison completion')
for name in ['protocol','gt-policy']:
    value = read(ROOT/('experiments/registry/datasets/iiit-cvid-spotting-test-261014-'+name+'.json'))
    display(HTML('<details><summary>Frozen '+escape(name)+'</summary><pre>'+escape(json.dumps(value,ensure_ascii=False,indent=2))+'</pre></details>'))

for candidate in registry["candidates"]:
    display(HTML("<details><summary>Locked candidate: "+escape(candidate["candidate_id"])+"</summary><pre>"+escape(json.dumps(candidate,ensure_ascii=False,indent=2))+"</pre></details>"))


## 2. Isolated orchestrator, source và model preparation
Không cài package vào kernel Colab. Bootstrap tránh host ensurepip; Python3.11 orchestrator/scorer và model venvs có seals riêng. Giữ đường dẫn driver sau CUDA/cuDNN wheels. Runtime mới có thể tải source và packages lớn; chưa có thời lượng chạy đo cho cả6 pipelines.


In [ ]:
from experiments.scripts.spotting_colab_environments import prepare_environment
try:
    PYTHON, orchestrator_seal = prepare_environment(dict(candidate_id='orchestrator',environment='scoring',packages=['pandas==2.2.3','matplotlib==3.10.1','huggingface_hub==0.30.2']),WORK)
except ValueError as reconstruction_error:
    if "offline reconstruction differs" in str(reconstruction_error):
        from experiments.scripts.spotting_colab_environments import environment_snapshot
        from experiments.scripts.spotting_colab_artifacts import digest
        def snapshot_difference(expected, actual):
            result = dict(python_changed=expected['python'] != actual['python'],
                          interpreter_changed=expected['executable_sha256'] != actual['executable_sha256'],
                          packages=[])
            original, rebuilt = expected['packages'], actual['packages']
            for name in sorted(set(original) | set(rebuilt)):
                before, after = original.get(name), rebuilt.get(name)
                if before == after:
                    continue
                row = dict(package=name, expected_version=(before or {}).get('version'),
                           actual_version=(after or {}).get('version'))
                files_before, files_after = (before or {}).get('files', {}), (after or {}).get('files', {})
                row['missing_files'] = sorted(set(files_before) - set(files_after))
                row['added_files'] = sorted(set(files_after) - set(files_before))
                row['changed_files'] = [dict(path=p, expected=files_before[p], actual=files_after[p])
                                        for p in sorted(set(files_before) & set(files_after)) if files_before[p] != files_after[p]]
                result['packages'].append(row)
            return result
        
        
        cache_root = Path(os.environ.get('SPOTTING_CACHE_ROOT',
                          '/content/drive/MyDrive/OCR/cvid-spotting-t4-261015-v2-r2-runtime-cache'))
        seal_path = cache_root / 'environments/orchestrator/environment-seal.json'
        python = Path('/content/cvid-spotting-t4/environments/orchestrator/venv/bin/python')
        assert seal_path.is_file(), 'Mount Drive and locate the r2 orchestrator cache seal first.'
        assert python.is_file(), 'Actual rebuilt environment is no longer present; use the updated notebook to capture the next reconstruction failure.'
        cached = json.loads(seal_path.read_text())
        actual = environment_snapshot(python)
        report = snapshot_difference(cached['installed'], actual)
        report.update(expected_digest=cached['installed_digest'], actual_digest=digest(actual),
                      cached_snapshot_self_consistent=digest(cached['installed']) == cached['installed_digest'])
        out = OUT
        out.mkdir(parents=True, exist_ok=True)
        report_path = out / 'orchestrator-reconstruction-diff.json'
        report_path.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding='utf-8')
        print(json.dumps(report, ensure_ascii=False, indent=2))
        assert json.loads(report_path.read_text(encoding='utf-8')) == report
        print('Saved:', report_path)
        files.download(str(report_path))  # Keep a local copy before disconnecting runtime.
    raise
print('Orchestrator:',PYTHON,'installed_digest:',orchestrator_seal['installed_digest'])
print('Orchestrator check finished. No dataset download, model install or inference has run in this cell.')


### Tiếp tục acquisition/model preparation sau khi orchestrator đạt
Nếu đang thu diff cache, dừng trước cell này. Ba code cells đầu chỉ mount/verify và kiểm orchestrator. Nếu mismatch, JSON được lưu lên Drive và gọi tải về máy trước khi báo lỗi; chưa có model inference.


In [ ]:
ARCHIVE = Path(SOURCE_ZIP) if SOURCE_ZIP else WORK / 'IIIT-CVid.zip'
if not ARCHIVE.exists():
    download_script = """from huggingface_hub import hf_hub_download, list_repo_files
import shutil, sys
repo = 'staghado/IIIT-CVid'
revision = '8159222f11460bff70bec7990ade19c468755fe4'
choices = [p for p in list_repo_files(repo, repo_type='dataset', revision=revision) if p.endswith('.zip')]
assert len(choices) == 1, choices
cached = hf_hub_download(repo, choices[0], repo_type='dataset', revision=revision)
shutil.copyfile(cached, sys.argv[1])
"""
    subprocess.run([str(PYTHON), '-c', download_script, str(ARCHIVE)], check=True)
if file_sha(ARCHIVE) != 'b5b562580e250ee3336462b7699bc79d0d1149d3b8cc82f1b221a0d0ce4fb489':
    raise ValueError('Source archive differs from frozen CVID release')
BASE = [str(PYTHON), '-m', 'experiments.scripts.run_spotting_colab',
        '--bundle-root', str(ROOT), '--work-root', str(WORK), '--output-root', str(OUT),
        '--candidate-registry', str(ROOT / 'experiments/registry/datasets/iiit-cvid-spotting-colab-t4-candidates-261015-v2.json'), '--source-zip', str(ARCHIVE)]

def tail_text(path):
    with Path(path).open('rb') as handle:
        handle.seek(max(0,Path(path).stat().st_size-12000))
        return handle.read().decode('utf-8',errors='replace')
def cli(command,candidate=None,check=True):
    import signal
    args = BASE+[command]+(['--candidate',candidate] if candidate else [])
    log = OUT/'orchestrator-logs'/(command+'-'+uuid.uuid4().hex+'.log')
    log.parent.mkdir(exist_ok=True)
    panel = display(HTML('<b>'+escape(command)+' RUNNING</b>'),display_id=True)
    with log.open('w') as stream:
        process = subprocess.Popen(args,cwd=ROOT,stdout=stream,stderr=subprocess.STDOUT,start_new_session=True)
        try:
            while process.poll() is None:
                text = tail_text(log)
                workers = list(OUT.glob('*/logs/*.log'))
                if workers:
                    latest = max(workers,key=lambda p:p.stat().st_mtime)
                    text += '\nLATEST WORKER: '+str(latest)+'\n'+tail_text(latest)
                panel.update(HTML('<b>'+escape(command)+' RUNNING</b><pre>'+escape(text[-18000:])+'</pre>'))
                time.sleep(3)
            code = process.wait()
        except KeyboardInterrupt:
            os.killpg(process.pid,signal.SIGTERM)
            try: process.wait(timeout=5)
            except subprocess.TimeoutExpired:
                os.killpg(process.pid,signal.SIGKILL)
                process.wait()
            raise
    panel.update(HTML('<b>'+escape(command)+' returncode='+str(code)+'</b><pre>'+escape(tail_text(log))+'</pre>'))
    table(status_rows(OUT,registry),'Candidate status after '+command)
    print('Full phase log:',log)
    if code!=0:
        phase = 'infer' if command=='run' else command
        for failed in status_rows(OUT,registry):
            if candidate and failed['candidate_id']!=candidate:
                continue
            if failed['latest_failure']:
                paths = list((OUT/failed['candidate_id']/'logs').glob(phase+'-*.log'))
                if paths:
                    path = max(paths,key=lambda p:p.stat().st_mtime)
                    display(HTML('<b>FAILED WORKER: '+escape(str(path))+'</b><pre>'+escape(tail_text(path))+'</pre>'))
                diagnostics = sorted((OUT/failed['candidate_id']/'diagnostics').glob('runtime-mismatch-'+phase+'-*.json'),key=lambda p:p.stat().st_mtime)
                if diagnostics:
                    diagnostic = read(diagnostics[-1])
                    print('Full runtime snapshots:',diagnostics[-1])
                    table(diagnostic['binding_diff'],'Binding differences')
                    table(diagnostic['runtime_diff'],'Runtime differences (no fields omitted)')
    if check and code!=0:
        raise RuntimeError('Phase failed; inspect logs/failures and export evidence. Do not change config.')
    return code
cli('prepare')


## 3. Data review và ALL6 preflight
Giữ selected IDs/GT:1.754/40.063,1 whole-frame geometry HOLD và97 metadata-only repairs. Models chỉ nhận full PNG. Preflight mới dùng9 train inputs, không test scores; qualification PGNet không được nhập làm gate. Mọi required gate phải PASS trước ANY test inference.


In [ ]:
data = read(OUT/'prepared-data.json')
assert len(data['frames'])==1754 and sum(r['object_count'] for r in data['frames'])==40063
table([dict(course=course,frames=sum(r['course']==course for r in data['frames']),gt=sum(r['object_count'] for r in data['frames'] if r['course']==course)) for course in sorted({r['course'] for r in data['frames']})], 'Frozen cohort by course')
table([dict(frame_id=r['frame_id'],course=r['course'],source_group=r['source_group'],gt=r['object_count'],decision=r['decision']) for r in data['frames']], 'ALL 1754 selected frame IDs')
table(data['train_preflight'],'Train preflight inputs')
preflight_order = sorted(registry['candidates'],key=lambda c:c['adapter']!='doctr')
for candidate in preflight_order:
    preflight_code = cli('preflight',candidate=candidate['candidate_id'],check=False)
    assert preflight_code == 0, 'Current candidate preflight failed; inspect the displayed worker traceback and export before test.'
require_preflights(OUT,registry)
print('PASS ALL6 preflight. Now replay train in fresh isolated processes; no test pixels.')
for candidate in preflight_order:
    replay_code = cli('verify',candidate=candidate['candidate_id'],check=False)
    assert replay_code == 0, 'Train replay binding failed; inspect diagnostics and export evidence before test.'
require_train_replays(OUT,registry)
print('PASS ALL6 preflight + separate-process train replay. Test is admitted.')


## 4. Full cohort — immutable checkpoints
Chỉ chạy khi ALL6 preflight và ALL6 separate-process train replay PASS. Mỗi candidate commit đủ1.754 frame; frame error giữ GT/FN và zero predictions. Không chạy lỗi bằng config khác, bỏ candidate hoặc bỏ frame. Cùng RUN_ID/bundle/runtime để resume; identity thay đổi phải dùng run mới. Worker giữ records đã commit. Không nhập CPU results cũ.


In [ ]:
require_train_replays(OUT,registry)
cli('run')
rows = status_rows(OUT,registry)
assert all(r['complete']=='complete' and r['committed_frames']==1754 for r in rows if r['required']), 'Incomplete six-candidate cohort; export evidence, no scores yet.'
print('Complete coverage: 10524 frame-candidate records, including recorded failures.')


## 5. Scoring, toàn bộ tables và plots
Pooled E2E Hmean chính; detection P/R/F1, TP/FP/FN, failure/invalid/empty, macro-course, course/folder tables. Fixed sensitivities: IoU>0.75, case-sensitive,3-frame influence subset (1.751/40.023), dùng cùng predictions. Exploratory paired bootstrap course/folder, không winner/generalization claim. Primary frame table đủ IDs; full transcription qua viewer và gz exports.


In [ ]:
cli('analyze')
subprocess.run([str(PYTHON),'-m','pip','check'],check=True)
render_env = dict(os.environ, MPLBACKEND='Agg')
subprocess.run([str(PYTHON),'-c','from experiments.scripts.spotting_colab_notebook import render_figures; import sys; render_figures(sys.argv[1])',str(OUT)],cwd=ROOT,env=render_env,check=True)
from experiments.scripts.spotting_colab_review import display_full_results
display_full_results(OUT)


## 6. Any-frame review: GT và sáu native predictions
Dropdown chứa đủ1.754 frame. Chọn rồi chạy viewer bên dưới; nếu không có widgets thì sửa FRAME_ID. Mặc định frame đầu cohort, không chọn theo điểm. GT xanh/prediction xanh dương không biểu thị TP/FP.


In [ ]:
FRAME_ID = data['frames'][0]['frame_id']
try:
    import ipywidgets as widgets
    FRAME_SELECTOR = widgets.Dropdown(options=[r['frame_id'] for r in data['frames']],value=FRAME_ID,description='Frame ID:',layout={'width':'95%'})
    display(FRAME_SELECTOR)
except ImportError:
    FRAME_SELECTOR = None
    print('Edit FRAME_ID:',FRAME_ID)


In [ ]:
from experiments.scripts.spotting_colab_frame_review import inspect_frame
selected_frame = FRAME_SELECTOR.value if FRAME_SELECTOR is not None else FRAME_ID
inspect_frame(OUT,ROOT,WORK,selected_frame)


In [ ]:
# Optional download of any complete export listed in analysis artifacts.
EXPORT_FILE = 'all-prediction-text.csv.gz'
summary = read(OUT/'analysis-summary.json')
assert EXPORT_FILE in summary['artifacts']
assert file_sha(OUT/EXPORT_FILE)==summary['artifacts'][EXPORT_FILE]['sha256']
files.download(str(OUT/EXPORT_FILE))


## 7. Export evidence — kể cả khi thiếu test/analysis
Cell cuối chạy riêng sau lỗi để lưu partial evidence. ZIP trên Drive chứa logs, prepared/runtime seals, checkpoints, completion và kết quả đã tạo. ZIP không chứng minh completion; analysis/gates mới quyết định trạng thái. File→Save rồi Download.ipynb để giữ outputs. Cache/wheels nằm ngoài evidence ZIP và được giữ trên Drive để resume.


In [ ]:
from experiments.scripts.spotting_colab_notebook import export_evidence
exported = export_evidence(OUT)
print(json.dumps(exported, indent=2))
files.download(exported['path'])
